In [17]:
import math
import pandas as pd
import numpy as np
import wandb

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

### functions

In [18]:
def get_wandb_table(
    tags,
    project="ideas_cv/llm-random-test",
    negative_tags=None,
    columns=None,
    print_columns=False,
):
    # WandB automatically looks for the WANDB_API_KEY environment variable.
    # No need to explicitly pass it like Neptune's token, but you must ensure it's exported in your env.
    api = wandb.Api()
    
    # Ensure tags is a list
    if isinstance(tags, str):
        tags = [tags]
        
    # Build WandB filter using MongoDB-like syntax.
    # $and ensures the run contains ALL of the specified positive tags (matching Neptune's default).
    filters = {"$and": [{"tags": tag} for tag in tags]} if tags else {}
    
    # Fetch runs via API
    runs = api.runs(path=project, filters=filters)
    
    # Extract data into a list of dictionaries to convert to pandas
    runs_data = []
    for run in runs:
        run_dict = {
            "sys/id": run.id,
            "sys/name": run.name,
            "sys/state": run.state,
            "sys/tags": run.tags,  # WandB tags are already a list of strings
        }
        # Flatten config (hyperparameters) and summary (metrics) into the dict.
        # Adding prefixes keeps the structure clean and prevents key collisions.
        run_dict.update({f"config/{k}": v for k, v in run.config.items()})
        run_dict.update({f"summary/{k}": v for k, v in run.summary._json_dict.items()})
        
        runs_data.append(run_dict)
        
    runs_table = pd.DataFrame(runs_data)
    
    if runs_table.empty:
        print("df shape: (0, 0)")
        return runs_table

    # Filter down to specific columns if provided
    if columns:
        # We must keep "sys/tags" temporarily if negative_tags are provided
        cols_to_keep = set(columns)
        if negative_tags:
            cols_to_keep.add("sys/tags")
        
        available_cols = [col for col in cols_to_keep if col in runs_table.columns]
        runs_table = runs_table[available_cols]

    # Apply negative tags filter
    if negative_tags:
        if isinstance(negative_tags, str):
            negative_tags = [negative_tags]
            
        for neg_tag in negative_tags:
            # Filter out rows where the negative tag exists in the list of tags
            runs_table = runs_table[
                ~runs_table["sys/tags"].apply(lambda x: neg_tag in x if isinstance(x, list) else False)
            ]

    # Print columns
    if print_columns:
        print("\n=== Available columns ===")
        for col in sorted(runs_table.columns):
            print(f"\t{col}")
        print("========================\n")

    print(f"df shape: {runs_table.shape}")

    return runs_table


def joint_tasks_plot_fetch(
    df: pd.DataFrame,
    task_keys: list,
    group_fn,
    project: str = "ideas_cv/llm-random-test",
):
    records = []
    api = wandb.Api()

    for _, row in df.iterrows():
        run_id = row["sys/id"]
        run = api.run(f"{project}/{run_id}")

        hist = run.history(
            keys=["steps/eval/loss"] + task_keys,
            pandas=True,
        )

        hist = hist.dropna(subset=["steps/eval/loss"]).copy()
        hist["_step"] = hist["_step"] - 1

        group_value = group_fn(row)
        dmodel = row["config/common"].get("dmodel") if isinstance(row.get("config/common"), dict) else None

        for _, hrow in hist.iterrows():
            loss = hrow["steps/eval/loss"]
            step = hrow["_step"]

            for task_key in task_keys:
                if task_key in hrow and pd.notna(hrow[task_key]):
                    records.append(
                        {
                            "run_id": run_id,
                            "group": group_value,
                            "step": step,
                            "loss": loss,
                            "task": task_key,
                            "task_value": hrow[task_key],
                            "dmodel": dmodel,
                        }
                    )

    return pd.DataFrame(records)


def _extract_group_value(row: pd.Series, group_col: str):
    """
    Supports:
      - direct dataframe columns, e.g. "config/model_dim"
      - nested lookup with dot notation, e.g. "config/common.sequence_length"
    """
    if group_col in row.index:
        return row[group_col]

    if "." in group_col:
        root, *rest = group_col.split(".")
        if root in row.index:
            value = row[root]
            for key in rest:
                if isinstance(value, dict):
                    value = value.get(key)
                else:
                    return None
            return value

    return None


def plot_joint_tasks(
    plot_df: pd.DataFrame,
    task_keys: list,
    plot_title: str,
    legend_title: str = "Group",
    height=None,
    width=1200,
    fit_lines=False,
):
    plot_df = plot_df.copy()
    plot_df["group"] = plot_df["group"].astype(str)
    plot_df["loss"] = plot_df["loss"].astype(float)
    plot_df["task_value"] = plot_df["task_value"].astype(float)

    # log-transform perplexity
    perp_mask = plot_df["task"] == "steps/eval/lambada_standard/perplexity"
    plot_df.loc[perp_mask, "task_value"] = np.log(plot_df.loc[perp_mask, "task_value"])

    # nicer facet titles
    plot_df["task_label"] = (
        plot_df["task"]
        .str.replace("steps/eval/", "", regex=False)
        .str.replace("lambada_standard/perplexity", "lambada (log perplexity)", regex=False)
        .str.replace("lambada_standard/acc", "lambada (acc)", regex=False)
        .str.replace("/acc_norm", "", regex=False)
        .str.replace("/acc", "", regex=False)
    )

    task_order = (
        plot_df[["task", "task_label"]]
        .drop_duplicates()
        .set_index("task")["task_label"]
        .reindex(task_keys)
        .tolist()
    )

    group_order = sorted(plot_df["group"].dropna().unique(), key=lambda x: (len(x), x))
    colors = px.colors.qualitative.Plotly
    color_map = {g: colors[i % len(colors)] for i, g in enumerate(group_order)}

    n_tasks = len(task_order)
    ncols = 2
    nrows = math.ceil(n_tasks / ncols)

    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=task_order,
        horizontal_spacing=0.08,
        vertical_spacing=0.12,
    )

    shown_legend = set()

    for i, task_label in enumerate(task_order):
        row = i // ncols + 1
        col = i % ncols + 1

        task_df = plot_df[plot_df["task_label"] == task_label]

        for group in group_order:
            sub = task_df[task_df["group"] == group].copy()
            if sub.empty:
                continue

            color = color_map[group]

            # points
            fig.add_trace(
                go.Scatter(
                    x=sub["loss"],
                    y=sub["task_value"],
                    mode="markers",
                    name=group,
                    legendgroup=group,
                    showlegend=group not in shown_legend,
                    marker=dict(size=6, color=color, opacity=0.8),
                    customdata=np.stack(
                        [
                            sub["dmodel"].astype(object).to_numpy(),
                            sub["step"].to_numpy(),
                        ],
                        axis=-1,
                    ),
                    hovertemplate=(
                        "loss=%{x}<br>"
                        "value=%{y}<br>"
                        "dmodel=%{customdata[0]}<br>"
                        "step=%{customdata[1]}<extra>%{fullData.name}</extra>"
                    ),
                ),
                row=row,
                col=col,
            )
            shown_legend.add(group)

            # fitted line
            if fit_lines and len(sub) >= 2:
                x = sub["loss"].to_numpy()
                y = sub["task_value"].to_numpy()

                mask = np.isfinite(x) & np.isfinite(y)
                x = x[mask]
                y = y[mask]

                if len(x) >= 2 and np.unique(x).size >= 2:
                    m, b = np.polyfit(x, y, 1)
                    xs = np.linspace(x.min(), x.max(), 100)
                    ys = m * xs + b

                    fig.add_trace(
                        go.Scatter(
                            x=xs,
                            y=ys,
                            mode="lines",
                            name=group,
                            legendgroup=group,
                            showlegend=False,
                            line=dict(color=color, width=2),
                            hoverinfo="skip",
                        ),
                        row=row,
                        col=col,
                    )

        fig.update_xaxes(title_text="Final Loss", row=row, col=col)
        fig.update_yaxes(title_text="", row=row, col=col)

    fig.update_layout(
        title=plot_title,
        width=width,
        height=height or 300 * nrows,
        legend_title_text=legend_title,
    )

    fig.show()

### plots

In [8]:
tags = ["2503_grid"]
negative_tags = ["remove"]
df = get_wandb_table(
    tags=tags,
    negative_tags=negative_tags
)
df = df[df['sys/state'] == 'finished']
print(f"df shape: {df.shape} (finished)")

task_keys = [
    "steps/eval/lambada_standard/perplexity",
    "steps/eval/lambada_standard/acc",
    "steps/eval/winogrande/acc",
    "steps/eval/social_iqa/acc",
    "steps/eval/sciq/acc_norm",
    "steps/eval/openbookqa/acc_norm",
    "steps/eval/hellaswag/acc_norm",
    "steps/eval/arc_easy/acc_norm",
    "steps/eval/arc_challenge/acc_norm",
    "steps/eval/piqa/acc_norm",
]

df shape: (25, 597)
df shape: (25, 597) (finished)


In [19]:
def mha_mqa_group(row):
    kv_heads = row["config/common"]["kv_heads"]
    return "mqa" if kv_heads == 1 else "mha"

plot_df = joint_tasks_plot_fetch(df, task_keys, group_fn=mha_mqa_group)

plot_df.to_csv('tmp.csv')
print("saved")
plot_joint_tasks(plot_df, task_keys, plot_title="MHA vs MQA", legend_title="attention type", fit_lines=True)

saved


In [20]:
def sequence_length_group(row):
    return row["config/common"]["sequence_length"]

plot_df = joint_tasks_plot_fetch(df, task_keys, group_fn=sequence_length_group)

plot_joint_tasks(plot_df, task_keys, plot_title="Sequence length", legend_title="sequence length", fit_lines=True)

In [26]:
seq_len = 1024

def mha_mqa_group(row):
    kv_heads = row["config/common"]["kv_heads"]
    return "mqa" if kv_heads == 1 else "mha"

sdf = df[df["config/common"].apply(lambda d: d.get("sequence_length") == seq_len)]
plot_sdf = joint_tasks_plot_fetch(sdf, task_keys, group_fn=mha_mqa_group)

plot_sdf = plot_sdf[plot_sdf['step'] != 320000]

plot_joint_tasks(plot_sdf, task_keys, plot_title=f"MHA vs MQA, seq len = {seq_len}", legend_title="attention type", fit_lines=True)